# Sprint 1 & 2 checkpoint: rental listings, map, and SA2 link

**MAST30034 Project 2, Group 55**

This notebook covers the Sprint 1 and Sprint 2 checkpoints:

1. Where our rental listing data comes from and how much of it we have
2. A first look at the data and basic cleaning
3. Maps of where the properties are (Sprint 1)
4. Linking every listing to its SA2 district (Sprint 2)
5. The external datasets we will use to answer the three project questions
6. How we are adding a second year of listings and a rent history

**Data source.** Rental listings from Domain.com.au, collected in September 2025 by the MAST30034 teaching team
(Wenqin Liu, Wenjie Wang, Mingming Gong) and released on the subject's Canvas page. The data is for MAST30034 coursework only
and is not stored in our GitHub repository.

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

# Paths are relative to this notebook's folder (notebooks/).
# Everything under data/raw and data/curated is kept off GitHub by .gitignore.
DATA_DIR = Path("../data/raw/domain/Data")                                # course dataset
SA2_ZIP = Path("../data/raw/external/SA2_2021_AUST_SHP_GDA2020.zip")      # ABS SA2 boundaries (2021)
OUT_DIR = Path("../data/curated")                                         # outputs that contain listing data
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Where the listing data comes from

In [ ]:
listings = pd.read_csv(DATA_DIR / "vic_rentals_all.csv")
suburb_summary = pd.read_csv(DATA_DIR / "suburb_summary.csv")
postcodes = pd.read_csv(DATA_DIR / "postcodes.csv")

website = listings["url"].str.extract(r"https?://([^/]+)/")[0].value_counts()
date_listed = pd.to_datetime(listings["date_listed"], errors="coerce")

print(f"Listings:                 {len(listings):,}")
print(f"Suburbs with listings:    {listings['suburb'].nunique():,} of {len(postcodes):,} Victorian suburbs")
print(f"Website:                  {', '.join(website.index)}")
print(f"Collected on:             {listings['scraped_date'].min()[:10]}")
print(f"Oldest 'date listed':     {date_listed.min():%Y-%m-%d}  (still on the market in Sept 2025)")
print(f"Cap per suburb:           {suburb_summary['listing_count'].max()} listings (15 pages x 20)")

All listings come from Domain.com.au and were collected on one day. This is a **single snapshot**: every row was
a live advert in September 2025. The `date_listed` column shows when each advert went up, not a past rent, so it cannot
give us rent history on its own. Section 6 explains how we add history.

Busy suburbs such as Melbourne, Southbank and Tarneit hit the 300-listing cap, so they are slightly under-counted.

## 2. First look and basic cleaning

In [ ]:
listings["weekly_rent"].describe().round(0)

In [ ]:
# Rents below $50 or above $20,000 a week are almost always typing errors or sale prices
bad_rent = listings["weekly_rent"].isna() | ~listings["weekly_rent"].between(50, 20_000)
print(f"Rows with missing or implausible rent: {bad_rent.sum():,} ({bad_rent.mean():.1%})")

clean = listings.loc[~bad_rent].copy()
clean = clean.dropna(subset=["lat", "lon"])
print(f"Rows kept for analysis: {len(clean):,}")

# Bond is not a useful feature: in Victoria it is set at one month's rent,
# so it just repeats the thing we are trying to predict.
ratio = (clean["bond"] / clean["weekly_rent"]).median()
print(f"Median bond / weekly rent = {ratio:.2f}  (one month's rent is 52 / 12 = {52/12:.2f} weeks)")

In [ ]:
by_type = (clean.groupby("property_type")["weekly_rent"]
                .agg(listings="size", median_rent="median")
                .sort_values("listings", ascending=False)
                .head(8))
by_beds = (clean[clean["bedrooms"].between(0, 5)]
                .groupby("bedrooms")["weekly_rent"]
                .agg(listings="size", median_rent="median"))
display(by_type)
display(by_beds)

In [ ]:
missing = listings.isna().mean().sort_values(ascending=False).head(8).rename("share missing")
missing.to_frame().style.format("{:.0%}")

Notes for the team:
- `land_area` is empty for every row, so land size will need another source if we want it.
- `carspaces` is missing for about 14% of rows and `structured_features` for about 10%.
- `structured_features` is a comma-separated list (e.g. "Pets Allowed, Dishwasher") that we can split into yes/no columns.

## 3. Where the properties are (Sprint 1 map)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
pts = ax.scatter(clean["lon"], clean["lat"], c=clean["weekly_rent"].clip(upper=1200),
                 s=3, cmap="viridis", alpha=0.6)
fig.colorbar(pts, ax=ax, label="Weekly rent ($, capped at 1,200 for colour)")
ax.set(title="Rental listings in Victoria, September 2025", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect("equal")
plt.show()

In [ ]:
# Interactive map: a heat layer of all listings, plus one circle per suburb
suburb_stats = (clean.groupby(["suburb", "postcode"])
                     .agg(listings=("listing_id", "size"),
                          median_rent=("weekly_rent", "median"),
                          lat=("lat", "median"), lon=("lon", "median"))
                     .reset_index())

m = folium.Map(location=[-37.81, 144.96], zoom_start=9)  # OpenStreetMap background
HeatMap(clean[["lat", "lon"]].values.tolist(), radius=8, blur=10,
        name="All listings (heat)").add_to(m)

colour = lambda rent: ("#2c7bb6" if rent < 450 else "#abd9e9" if rent < 550
                       else "#fdae61" if rent < 700 else "#d7191c")
suburb_layer = folium.FeatureGroup(name="Suburb median rent")
for r in suburb_stats.itertuples():
    folium.CircleMarker(
        location=[r.lat, r.lon], radius=3 + r.listings ** 0.5 / 2,
        color=colour(r.median_rent), fill=True, fill_opacity=0.8, weight=0,
        popup=f"{r.suburb.title()} {r.postcode}<br>Median rent: ${r.median_rent:,.0f}/wk<br>Listings: {r.listings}",
    ).add_to(suburb_layer)
suburb_layer.add_to(m)
folium.LayerControl().add_to(m)

m.save(OUT_DIR / "listings_map.html")
m

Circle colour shows the suburb's median weekly rent (blue under \$450, light blue \$450–549, orange \$550–699, red \$700+),
and circle size shows how many listings the suburb has. The map is also saved to `data/curated/listings_map.html` (not committed, because it contains every listing's location).

## 4. Linking listings to SA2 districts (Sprint 2)

Population forecasts (Victoria in Future 2023) and income (2021 Census) are published by **SA2 district**, so each
listing needs an SA2 code. We use the ABS **2021** SA2 boundaries because both of those datasets use the 2021 districts.

Download: ABS → *Australian Statistical Geography Standard (ASGS) Edition 3* → *Digital boundary files* →
"Statistical Areas Level 2 - 2021 - Shapefile" (GDA2020). Put the zip file in `data/raw/external/`; no need to unzip it.

In [ ]:
def attach_sa2(df, sa2_path):
    """Add SA2 code and name to each listing by checking which district its point falls within."""
    sa2 = gpd.read_file(sa2_path)
    sa2 = sa2[sa2["STE_NAME21"] == "Victoria"].dropna(subset=["geometry"])
    points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    points = points.to_crs(sa2.crs)
    joined = gpd.sjoin(points, sa2[["SA2_CODE21", "SA2_NAME21", "geometry"]],
                       how="left", predicate="within").drop(columns="index_right")
    return joined, sa2

if SA2_ZIP.exists():
    listings_sa2, sa2_vic = attach_sa2(clean, SA2_ZIP)
    matched = listings_sa2["SA2_CODE21"].notna().mean()
    print(f"Listings matched to an SA2: {matched:.1%}")
    print(f"SA2 districts with at least one listing: {listings_sa2['SA2_CODE21'].nunique()} of {len(sa2_vic)}")

    listings_sa2.drop(columns="geometry").to_csv(OUT_DIR / "listings_with_sa2.csv", index=False)
else:
    print(f"SA2 file not found at {SA2_ZIP}. Download it (see above) and re-run this cell.")

In [ ]:
if SA2_ZIP.exists():
    sa2_rent = (listings_sa2.groupby("SA2_CODE21")["weekly_rent"]
                .agg(listings="size", median_rent="median").reset_index())
    sa2_map = sa2_vic.merge(sa2_rent, on="SA2_CODE21", how="left")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sa2_map.plot(column="median_rent", ax=axes[0], legend=True, cmap="viridis",
                 missing_kwds={"color": "lightgrey"},
                 legend_kwds={"label": "Median weekly rent ($)"})
    axes[0].set_title("Median weekly rent by SA2 (grey = no listings)")

    greater_melb = sa2_map.cx[144.4:145.6, -38.5:-37.5]
    greater_melb.plot(column="median_rent", ax=axes[1], legend=True, cmap="viridis",
                      missing_kwds={"color": "lightgrey"})
    axes[1].set_title("Greater Melbourne close-up")
    for ax in axes:
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()

## 5. External datasets

These are the outside datasets we plan to join to the listings, and which project question each one helps with.
(Q1 = most important features, Q2 = top 10 growth suburbs, Q3 = most liveable and affordable suburbs.)

| Dataset | Publisher | Level | What we get from it | Questions |
|---|---|---|---|---|
| SA2 boundaries, 2021 (ASGS Edition 3) | ABS | SA2 | District shapes, used to link listings to all SA2 data | All |
| Victoria in Future 2023: population projections | Vic. Dept of Transport and Planning | SA2 | Population to 2036 → population growth rate | Q2 |
| 2021 Census General Community Profile, table G02 | ABS | SA2 | Median weekly personal and household income → affluence | Q1, Q3 |
| Rental Report: moving annual rent by suburb | Vic. Dept of Families, Fairness and Housing | Suburb groups | Quarterly median rents going back many years → rent history for forecasting | Q2 |
| PTV train stations / GTFS timetable data | Transport Victoria (DataVic) | Point | Closest train station (Sprint 3 distances) | Q1, Q3 |
| School locations | Vic. Dept of Education (DataVic) | Point | Nearest schools | Q1, Q3 |
| OpenStreetMap (parks, shops, hospitals) + OpenRouteService | OSM contributors | Point / routes | Nearby amenities and driving distance to the CBD | Q1, Q3 |

## 6. Adding a second year of listings and a rent history

**Why we can't scrape 2024 listings.** Rental websites only show adverts that are live today. Once a property is leased
its advert disappears, so there is no way to scrape what was listed in 2024.

**What we are doing instead:**

1. **A new September 2026 snapshot from rent.com.au.** We collect listings for the same 656 suburbs with
   `scripts/scrape_rentcomau.py`, capped at 300 per suburb like the course dataset. It works in two stages: suburb search pages
   (rent, bedrooms, bathrooms, car spaces, property type) and then each listing's own page (bond, available date,
   features, walk and transit scores, coordinates if shown).
   - *How we scrape responsibly:* the script checks robots.txt first, waits about 5 seconds between requests,
     names itself as a student project, stops immediately if the site refuses access, and does not collect agents'
     names. The data stays on our computers and is not committed to GitHub.
   - *Things to keep in mind when comparing with the 2025 data:* it is a different website, so some listings appear
     on one site and not the other, and the columns are not identical (rent.com.au has walk scores but no
     "date listed"). We record the source of every row.
2. **Rent history from the Victorian Government Rental Report.** It is based on every bond lodged with the Residential
   Tenancies Bond Authority, and gives quarterly median rents by suburb group over many years. This is the data we will use
   for the 5-year rent forecasts in Sprint 4.

## 7. Next steps (Sprint 3)

- Finish the Sept 2026 collection and merge it with the 2025 data (add a `snapshot` column).
- Join population growth (VIF2023) and income (Census G02) to each listing by SA2 code.
- Work out driving distance to the nearest train station and to the Melbourne CBD with OpenRouteService.
- Split `structured_features` into yes/no columns and start listing which features go with higher rent.